<a href="https://colab.research.google.com/github/hyhyhy1105-cyber/selfish-spoon/blob/main/week04_assignment_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [1]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


torch 2.11.0+cpu / device: cpu


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [2]:
# 답안
import torch
import pandas as pd

# 1. 서로 다른 Tensor 5개 생성 (주석으로 이유 작성)
# t1: 스칼라(0차원) 데이터를 다루기 위한 텐서
t1 = torch.tensor(3.14)

# t2: 1차원 정수형 레이블/벡터 데이터를 표현하기 위한 텐서
t2 = torch.tensor([1, 2, 3, 4, 5], dtype=torch.int32)

# t3: 가중치(Weight) 행렬 초기화를 위한 2차원 난수 텐서
t3 = torch.randn(3, 4)

# t4: 배치 단위 이미지 데이터(배치, 채널, 높이 등) 형태를 다루기 위한 3차원 0-텐서
t4 = torch.zeros((2, 3, 3), dtype=torch.float64)

# t5: 등간격 수열 데이터를 얻기 위한 1차원 범위 텐서
t5 = torch.arange(0, 10, 2)

tensors = [("t1 (Scalar)", t1),
           ("t2 (Vector)", t2),
           ("t3 (Matrix)", t3),
           ("t4 (3D Tensor)", t4),
           ("t5 (Arange)", t5)]

# 2. 속성 확인표 생성 (Colab 데이터프레임 시각화 활용)
data = []
for name, t in tensors:
    data.append({
        "이름": name,
        "shape": str(list(t.shape)),
        "dim": t.ndim,
        "dtype": str(t.dtype).replace("torch.", ""),
        "device": str(t.device)
    })

df = pd.DataFrame(data)
display(df) # Colab에서 깔끔한 표로 출력됩니다.

,이름,shape,dim,dtype,device
0,t1 (Scalar),[],0,float32,cpu
1,t2 (Vector),[5],1,int32,cpu
2,t3 (Matrix),"[3, 4]",2,float32,cpu
3,t4 (3D Tensor),"[2, 3, 3]",3,float64,cpu
4,t5 (Arange),[5],1,int64,cpu


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [3]:
# 답안
import torch

# (3, 4)와 (4, 2) 크기의 Tensor 생성
A = torch.randn(3, 4)
B = torch.randn(4, 2)

# 1. 행렬곱 결과와 그 shape
matmul_result = torch.matmul(A, B) # 또는 A @ B
print("=== 1. 행렬곱 결과 ===")
print(matmul_result)
print("행렬곱 결과 shape:", matmul_result.shape)
print()

# 2. 첫 번째 Tensor(A)의 열별 평균과 행별 합계
col_mean = A.mean(dim=0) # dim=0: 열 기준 평균
row_sum = A.sum(dim=1)   # dim=1: 행 기준 합계

print("=== 2. 첫 번째 Tensor(A) 연산 ===")
print("열별 평균 (dim=0):", col_mean)
print("행별 합계 (dim=1):", row_sum)

=== 1. 행렬곱 결과 ===
tensor([[-2.7617,  0.8363],
        [ 0.1626, -2.3593],
        [ 2.3492,  3.9017]])
행렬곱 결과 shape: torch.Size([3, 2])

=== 2. 첫 번째 Tensor(A) 연산 ===
열별 평균 (dim=0): tensor([-0.2080, -0.3396, -0.0059,  0.4480])
행별 합계 (dim=1): tensor([-3.0221,  4.0676, -1.3621])


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [1]:
# 답안
import torch
import time

# 비교용 대형 행렬 생성
size = (5000, 5000)
A_cpu = torch.randn(size)
B_cpu = torch.randn(size)

# CPU 연산 시간 측정
start_time = time.time()
C_cpu = torch.matmul(A_cpu, B_cpu)
cpu_time = time.time() - start_time
print(f"CPU 실행 시간: {cpu_time:.4f} 초")

# GPU 사용 가능 여부 확인 후 연산
if torch.cuda.is_available():
    device = torch.device("cuda")
    A_gpu = A_cpu.to(device)
    B_gpu = B_cpu.to(device)

    # GPU 비동기 연산 특성을 맞춰주기 위한 동기화 코드
    torch.cuda.synchronize()
    start_time = time.time()
    C_gpu = torch.matmul(A_gpu, B_gpu)
    torch.cuda.synchronize()
    gpu_time = time.time() - start_time

    print(f"GPU 실행 시간: {gpu_time:.4f} 초")
    print(f"\n[차이 설명]\nGPU는 수천 개의 코어가 동시 병렬 처리하므로 대규모 행렬곱 연산에서 CPU 대비 훨씬 빠른 속도를 보입니다.")
else:
    print("\n[GPU 미사용 상태 안내]")
    print("GPU 실행 시간: N/A")
    print("이유: 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 가속기를 'T4 GPU'로 설정하지 않아 CPU 모드로 작동 중입니다.")

CPU 실행 시간: 2.5587 초
GPU 실행 시간: 0.1685 초

[차이 설명]
GPU는 수천 개의 코어가 동시 병렬 처리하므로 대규모 행렬곱 연산에서 CPU 대비 훨씬 빠른 속도를 보입니다.


## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [1]:
# 답안
import torch

# 1. y = 2x^3 + 5x 에서 x = 3 일 때 도함수
x1 = torch.tensor(3.0, requires_grad=True)
y1 = 2 * (x1 ** 3) + 5 * x1

# PyTorch 자동미분
y1.backward()
auto_grad1 = x1.grad.item()

# 손계산: dy/dx = 6x^2 + 5 -> x=3 대입 시 6*(3^2) + 5 = 59
manual_grad1 = 6 * (3.0 ** 2) + 5

print("=== 1번 문제 ===")
print(f"자동미분 결과: {auto_grad1}")
print(f"손계산 결과  : {manual_grad1}")
print(f"일치 여부    : {auto_grad1 == manual_grad1}\n")


# 2. 임의 수식: y = x^2 + 4x + 1 에서 x = 5 일 때
x2 = torch.tensor(5.0, requires_grad=True)
y2 = (x2 ** 2) + 4 * x2 + 1

# PyTorch 자동미분
y2.backward()
auto_grad2 = x2.grad.item()

# 손계산: dy/dx = 2x + 4 -> x=5 대입 시 2*(5) + 4 = 14
manual_grad2 = 2 * 5.0 + 4

print("=== 2번 문제 ===")
print("수식: y = x^2 + 4x + 1 (x = 5)")
print(f"자동미분 결과: {auto_grad2}")
print(f"손계산 결과  : {manual_grad2}")
print(f"일치 여부    : {auto_grad2 == manual_grad2}")

=== 1번 문제 ===
자동미분 결과: 59.0
손계산 결과  : 59.0
일치 여부    : True

=== 2번 문제 ===
수식: y = x^2 + 4x + 1 (x = 5)
자동미분 결과: 14.0
손계산 결과  : 14.0
일치 여부    : True
